In [ ]:
import yt_dlp
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter


class YTVideoFetcher:
    def __init__(self, topic, embedding_model, k=5):
        self.embedding_model = embedding_model
        
        self.topic = topic
        
        self.k = k
        self.imp_params = ['title', 'id', 'description', 'duration', 'view_count', 'like_count', 'webpage_url']
        self.url = f"ytsearch{k}:{self.topic}"
        self.ydl = yt_dlp.YoutubeDL({
            "queit":True
        })
        self.results = []
        self.search_videos()
        self.metadata = self.extract_metadata()

        

    def search_videos(self):

        self.results = self.ydl.extract_info(
            self.url,
            download=False
        )
    
    def extract_metadata(self):

        metadata = []

        for video in self.results["entries"]:

            video_data = {}

            for param in self.imp_params:
                video_data[param] = video.get(param)

            metadata.append(video_data)

        return metadata
    
    def filter_docs(self):
        topic_embedding = self.embedding_model.embed_query(
            self.topic
        )

        texts = [
            f"{doc['title']} {doc['description'][:500]}"
            for doc in self.metadata
        ]

        doc_embeddings = self.embedding_model.embed_documents(
            texts
        )

        for doc, emb in zip(self.metadata, doc_embeddings):
            doc["semantic_score"] = cosine_similarity(
                [topic_embedding],
                [emb]
            )[0][0]

        self.metadata.sort(
            key=lambda x: x["semantic_score"],
            reverse=True
        )

        self.metadata = self.metadata[: max(1, self.k // 2)]

        self.metadata.sort(
            key=lambda x: (
                x.get("view_count", 0),
                x.get("like_count", 0) or 0
            ),
            reverse=True
        )

        return self.metadata

In [ ]:
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-l6-v2")

In [ ]:
ytf = YTVideoFetcher("docker", k=2, embedding_model=embedding_model)
ytf.metadata

In [ ]:
ytf.metadata

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

In [ ]:
def get_transcripts():
    for video in ytf.metadata:
        id = video['id']
        transcripts.append(ytt)

In [ ]:
transcripts = []

for video in ytf.metadata:
    id = video['id']
    transcripts.append(ytt_api.fetch(id))


In [ ]:
class 

In [ ]:


response = ytt_api.fetch('3c-iBn73dDE')

In [ ]:
def chunk_transcript(
    transcript,
    max_chars=500
):
    chunks = []

    current_text = []
    start_time = None
    current_len = 0

    for seg in transcript:
        text = seg.text

        if start_time is None:
            start_time = seg.start

        if current_len + len(text) > max_chars:

            end_time = seg.start

            chunks.append({
                "content": " ".join(current_text),
                "start_time": start_time,
                "end_time": end_time
            })

            current_text = []
            start_time = seg.start
            
            current_len = 0

        current_text.append(text)
        current_len += len(text)

    return chunks

In [ ]:
chunks = chunk_transcript(transcripts[0], max_chars=400)

In [ ]:
len(chunks)

In [ ]:
from langchain_classic.schema import Document

docs = [Document(page_content=chunk['content'], metadata={"start_time": chunk['start_time'], "end_time":chunk['end_time']}) for chunk in chunks]



In [ ]:
docs[:5]



In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(docs, embedding_model)

In [ ]:
dimension = vector_store.index.d
num_vectors = vector_store.index.ntotal

size_bytes = dimension * num_vectors * 4
print(size_bytes / (1024**2), "MB")

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k":5}
)

In [ ]:
docs = retriever.invoke(
    "GTA 5"
)

for doc in docs:
    print(doc.page_content)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

query = "what is explained in the video"

docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

In [ ]:
from pydantic import BaseModel, Field
class schema(BaseModel):
    subtopics: list[str] = Field(
        description="A list of subtopics related to user's topic, each topic shoud be concise and suitable as a section in study notes."
    )



In [ ]:
structured_llm = llm.with_structured_output(schema)


In [ ]:
subtopics = structured_llm.invoke("docker").subtopics

In [ ]:
from langchain_classic.agents import create_tool_calling_agent

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
import wikipedia
wrapper = WikipediaAPIWrapper(
    top_k_results=5,            
    doc_content_chars_max=100, 
    lang="en"                   
)

tool = WikipediaQueryRun(api_wrapper=wrapper)


for i in subtopics:

    results = wikipedia.search(i)
    print(results)

In [ ]:
seen_titles = set()


for subtopic in subtopics:
    results = wikipedia.search(subtopic)

    for result in results:
        page = wikipedia.page(result)

        if page.title not in seen_titles:
            seen_titles.add(page.title)
            break

In [ ]:
from langchain_community.document_loaders import YoutubeLoader


loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=Gjnup-PuquQ&t=4s&pp=ygUPZmlyZXNoaXAgZG9ja2Vy"
)

In [ ]:
docs = loader.load()

In [ ]:
docs

In [ ]:
import os, requests
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('TRANSCRIPT_API_KEY')

API_KEY


In [ ]:
def get_transcripts(video_id):
    url = 'https://transcriptapi.com/api/v2/youtube/transcript'
    params = {'video_url': video_id, 'format': 'json'}
    r = requests.get(url, params=params, headers={'Authorization': f'Bearer {API_KEY}'}, timeout=30)
    r.raise_for_status()
    return r.json()['transcript']
    

In [ ]:
trans=get_transcripts("I3O-zmuFJdE")

In [ ]:
trans

In [ ]:
def chunk_transcript( transcript, video_id, max_chars=500):
    chunks = []

    current_text = []
    start_time = None
    current_len = 0

    for seg in transcript:
        text = seg["text"]

        if start_time is None:
            start_time = seg["start"]

        if current_len + len(text) > max_chars and current_text:

            end_time = seg["start"]

            chunks.append({
                "content": " ".join(current_text),
                "start_time": start_time,
                "end_time": end_time,
                "video_id": video_id
            })

            current_text = []
            start_time = seg["start"]
            current_len = 0

        current_text.append(text)
        current_len += len(text)

    if current_text:
        last_seg = transcript[-1]

        chunks.append({
            "text": " ".join(current_text),
            "start_time": start_time,
            "end_time": last_seg["start"] + last_seg["duration"],
            "video_id": video_id
        })

    return chunks

In [ ]:
chunks = chunk_transcript(transcript=trans, video_id="I3O-zmuFJdE", max_chars=100)

In [ ]:
from langchain_classic.schema import Document

In [ ]:
chunks[0]

In [ ]:
docs = [Document(page_content=chunk['content'], metadata={'start':chunk['start_time'], 'end_time':chunk['end_time'], 'video_id':chunk['video_id']}) for chunk in chunks]

In [ ]:
docs

In [ ]:
import yt_dlp
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter
from youtube_transcript_api import YouTubeTranscriptApi
import requests
import os 
from dotenv import load_dotenv

load_dotenv()


API_KEY = os.getenv("TRANSCRIPT_API_KEY")


class YTVideoFetcher:
    def __init__(self, topic, embedding_model, k=5):
        self.embedding_model = embedding_model
        
        self.topic = topic
        
        self.k = k
        self.imp_params = ['title', 'id', 'description', 'duration', 'view_count', 'like_count', 'webpage_url']
        self.url = f"ytsearch{k}:{self.topic}"
        self.ydl = yt_dlp.YoutubeDL({
            "queit":True
        })
        self.results = []
        self.search_videos()
        self.metadata = self.extract_metadata()

        

        

        

    def search_videos(self):

        self.results = self.ydl.extract_info(
            self.url,
            download=False
        )
    
    def extract_metadata(self):

        metadata = []

        for video in self.results["entries"]:

            video_data = {}

            for param in self.imp_params:
                video_data[param] = video.get(param)

            metadata.append(video_data)

        return metadata
    
    def filter_docs(self):
        topic_embedding = self.embedding_model.embed_query(
            self.topic
        )

        texts = [
            f"{doc['title']} {doc['description'][:500]}"
            for doc in self.metadata
        ]

        doc_embeddings = self.embedding_model.embed_documents(
            texts
        )

        for doc, emb in zip(self.metadata, doc_embeddings):
            doc["semantic_score"] = cosine_similarity(
                [topic_embedding],
                [emb]
            )[0][0]

        self.metadata.sort(
            key=lambda x: x["semantic_score"],
            reverse=True
        )

        self.metadata = self.metadata[: max(1, self.k // 2)]

        self.metadata.sort(
            key=lambda x: (
                x.get("view_count", 0),
                x.get("like_count", 0) or 0
            ),
            reverse=True
        )

        return self.metadata
    

    
    
    
    

    def get_transcripts(self)->dict:
        data = {}
        for video in self.metadata:

            video_id = video['id']
            
            url = 'https://transcriptapi.com/api/v2/youtube/transcript'
            params = {'video_url': video_id, 'format': 'json'}
            r = requests.get(url, params=params, headers={'Authorization': f'Bearer {API_KEY}'}, timeout=30)
            r.raise_for_status()
            transcript = r.json()['transcript']
            
            data[video_id] = transcript
        
        return data
        
        


    def chunk_transcript(self,transcript,id,max_chars=500):
        chunks = []

        current_text = []
        start_time = None
        current_len = 0

        for seg in transcript:
            text = seg['text']

            if start_time is None:
                start_time = seg['start']

            if current_len + len(text) > max_chars:

                end_time = seg['start']

                chunks.append({
                    "text": " ".join(current_text),
                    "start_time": start_time,
                    "end_time": end_time,
                    "video_id": id

                })

                current_text = []
                start_time = seg.start
                
                current_len = 0

            current_text.append(text)
            current_len += len(text)

        return chunks

    
    def transcriber_chunker(self):
        chunks = []

        data = self.get_transcripts()

        for id, transcript  in data.items():


            chunks.extend(self.chunk_transcript(id=id, transcript=transcript))
        
        return chunks



In [ ]:
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-l6-v2")

In [ ]:
ytf = YTVideoFetcher(topic="docker", embedding_model=embedding_model, k=3)

In [ ]:
transcripts = ytf.get_transcripts()

In [ ]:
transcripts

In [ ]:
ytf.chunk_transcript(transcript=transcripts[transcripts.keys()[0]], id=(list(transcripts.keys()))[0])

In [ ]:
import yt_dlp
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter
from youtube_transcript_api import YouTubeTranscriptApi
import requests
import os 
from dotenv import load_dotenv

load_dotenv()


API_KEY = os.getenv("TRANSCRIPT_API_KEY")


class YTVideoFetcher:
    def __init__(self, topic, embedding_model, k=5):
        self.embedding_model = embedding_model
        
        self.topic = topic
        
        self.k = k
        self.imp_params = ['title', 'id', 'description', 'duration', 'view_count', 'like_count', 'webpage_url', 'language']
        self.url = f"ytsearch{k}:{self.topic}"
        self.ydl = yt_dlp.YoutubeDL({
            "queit":True
        })
        self.results = []
        self.search_videos()
        self.metadata = self.extract_metadata()

        
    def language_check(self):
        for video_data in self.metadata
        

        

    def search_videos(self):

        self.results = self.ydl.extract_info(
            self.url,
            download=False
        )
    
    def extract_metadata(self):

        metadata = []

        for video in self.results["entries"]:

            video_data = {}

            for param in self.imp_params:
                video_data[param] = video.get(param)

            metadata.append(video_data)

        return metadata
    
    def filter_docs(self):
        topic_embedding = self.embedding_model.embed_query(
            self.topic
        )

        texts = [
            f"{doc['title']} {doc['description'][:500]}"
            for doc in self.metadata
        ]

        doc_embeddings = self.embedding_model.embed_documents(
            texts
        )

        for doc, emb in zip(self.metadata, doc_embeddings):
            doc["semantic_score"] = cosine_similarity(
                [topic_embedding],
                [emb]
            )[0][0]

        self.metadata.sort(
            key=lambda x: x["semantic_score"],
            reverse=True
        )

        self.metadata = self.metadata[: max(1, self.k // 2)]

        self.metadata.sort(
            key=lambda x: (
                x.get("view_count", 0),
                x.get("like_count", 0) or 0
            ),
            reverse=True
        )

        return self.metadata
    

    
    
    
    

    def get_transcripts(self)->dict:
        data = {}
        for video in self.metadata:

            video_id = video['id']
            
            url = 'https://transcriptapi.com/api/v2/youtube/transcript'
            params = {'video_url': video_id, 'format': 'json'}
            r = requests.get(url, params=params, headers={'Authorization': f'Bearer {API_KEY}'}, timeout=30)
            r.raise_for_status()
            transcript = r.json()['transcript']
            
            data[video_id] = transcript
        
        return data
        
        

    def chunk_transcript(self, data, max_chars=500):
        chunks = []

        for video_id, transcript in data.items():

            current_text = []
            start_time = None
            current_len = 0

            for seg in transcript:

                text = seg["text"]

                if start_time is None:
                    start_time = seg["start"]

                if current_len + len(text) > max_chars and current_text:

                    chunks.append({
                        "text": " ".join(current_text),
                        "start_time": start_time,
                        "end_time": seg["start"],
                        "video_id": video_id
                    })

                    current_text = []
                    start_time = seg["start"]
                    current_len = 0

                current_text.append(text)
                current_len += len(text)

            # Store the final chunk of this video
            if current_text:
                last_seg = transcript[-1]

                chunks.append({
                    "text": " ".join(current_text),
                    "start_time": start_time,
                    "end_time": last_seg["start"] + last_seg["duration"],
                    "video_id": video_id
                })

        return chunks
    
    def transcriber_chunker(self):
        data = self.get_transcripts()
        return self.chunk_transcript(data)


In [ ]:
ytf = YTVideoFetcher(topic="docker", embedding_model=embedding_model, k=3)

In [ ]:
ytf.metadata

In [ ]:
chunks = ytf.transcriber_chunker()

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient



# client = QdrantClient(":memory:")  -->>> for in memory storage of vectors


client = QdrantClient(host="localhost",port=6333)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="YoutubeTranscripts",
    embedding=embedding_model
)

In [ ]:
small_chunks = chunks[:10]

type(small_chunks[0])

In [ ]:
small_chunks[0].keys()

In [ ]:
docs = [Document(page_content=chunk['text'], metadata={'start_time':chunk['start_time'], 'end_time':chunk['end_time'], 'video_id':chunk['video_id']}) for chunk in small_chunks]

In [ ]:
from uuid import uuid4

ids = [str(uuid4()) for _ in range(len(docs))]

In [ ]:
vector_store.add_documents(documents=docs, ids=ids)

In [ ]:
vector_store.delete(ids=ids)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-l6-v2")

In [ ]:
ytf2 = YTVideoFetcher(topic="docker", embedding_model=embedding_model, k=2)

In [ ]:
topic = "docker"
k = 10
url = f"ytsearch{k}:{topic}"

In [ ]:

ydl = yt_dlp.YoutubeDL({})

info = ydl.extract_info(url, download=False)

In [ ]:
len(ytf.results['entries'])

In [ ]:
len(ytf.results['entries'])

In [ ]:
results = info

In [ ]:
len(results['entries'])

In [ ]:
results["entries"] = [
    video
    for video in results["entries"]
    if video.get("language") == "en"
]

In [ ]:
len(results['entries'])

In [ ]:
[video['language'] for video in results['entries']]

In [ ]:
len(results['entries'])

In [ ]:
def lang_check(self):
        for video_data in self.results['entries']:
            lang = video_data['language']
            if lang!='en':
                self.results['entries'].remove(video_data)

In [ ]:
from langchain_tavily import TavilySearch
from langchain_groq import ChatGroq
from langchain.agents import create_agent
import os

key = os.getenv("GROQ_API_KEY")

tv_search = TavilySearch(max_results=5, topic="general")

llm = ChatGroq(api_key=key, model="llama-3.3-70b-versatile")

agent = llm.bind_tools(tools=[tv_search])

In [ ]:
query = llm.invoke("provide prompt to give to tavily search for a person to learn docker online no instructor")

In [ ]:
query

In [ ]:
response = agent.invoke("Find online tutorials, courses, and resources to learn Docker on my own,")

In [ ]:
response.tool_calls

In [ ]:
tool_result = tv_search.invoke(response.tool_calls[0]['args'])

In [ ]:
len(tool_result['results'])

In [ ]:
tool_result['results']

In [ ]:
tool_result['results'][0]['content']

In [ ]:
import arxiv


search = arxiv.Search(
    query="quantum",
    max_results=4,
    sort_by=arxiv.SortCriterion.SubmittedDate
)

results = client.results(search)



In [ ]:
from semanticscholar import SemanticScholar

sch = SemanticScholar()

results = sch.search_paper(query='memory management', limit=3)

for paper in results.items:
    print(paper.title) 




In [ ]:
import trafilatura
url = "https://transformer-circuits.pub/2025/attention-qk/index.html"
downloaded = trafilatura.fetch_url(url)
text = trafilatura.extract(downloaded)

In [ ]:
len(text)

In [ ]:
import trafilatura
from langchain_text_splitters import RecursiveCharacterTextSplitter

class BlogPipeline:
    def __init__(self):
        
        pass

        


    def fetch_blog(self, url):
        downloaded = trafilatura.fetch_url(url)
        text = trafilatura.extract(downloaded)

        return text

    def chunker(self, text):
        splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

        chunks = splitter.split_text(text)

        return chunks
    

    def fetch_and_chunk(self, url):

        text = self.fetch_blog(url)

        chunks = self.chunker(text=text)

        return chunks
    






In [ ]:
bg = BlogPipeline()
bg.fetch_and_chunk("https://transformer-circuits.pub/2025/attention-qk/index.html")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-l6-v2")

In [ ]:

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient




client = QdrantClient(host="localhost",port=6333)

blog_vector_store = QdrantVectorStore(
    client=client,
    collection_name="BlogVectors",
    embedding=embedding_model
)



In [ ]:
info = client.get_collection("BlogVectors")

print(info.config.params.vectors)

In [ ]:
info = client.get_collection("BlogVectors")

print("Points:", info.points_count)
# print("Vectors count:", info.vectors_count)
print("Indexed vectors:", info.indexed_vectors_count)

In [ ]:
points, _ = client.scroll(
    collection_name="BlogVectors",
    limit=1,
    with_payload=True,
    with_vectors=False
)

print(points[0])

In [ ]:
info = client.get_collection("BlogVectors")

print(info)